In [ ]:
from __future__ import annotations

import datetime as _dt
import json
import os
import re
import shutil
import uuid
import zipfile
from pathlib import Path

In [ ]:
data_dir = "../data"
output_dir = os.path.join(data_dir, "1_parser")
attachments_dir = os.path.join(output_dir, "attachments")
os.makedirs(attachments_dir, exist_ok=True)

whatsapp_backup_path = os.path.join(data_dir, "whatsapp_backup.zip")
# whatsapp_backup_path = os.path.join(data_dir, "whatsapp_backup_slim.zip")
whatsapp_backup_extraction_path = os.path.join(data_dir, "1_parser" ,"whatsapp_backup_extracted")


In [ ]:
BRACKET_LINE_RE = re.compile(r"^\u200e?\[([^\]]+)\]\s([^:]+?):\s?(.*)$")
ATTACHMENT_RE = re.compile(r"<Anhang:\s([^>]+)>", re.IGNORECASE)
CALL_KEYWORDS = ("voice call", "video call", "sprachanruf", "videoanruf", "anruf")
TIMESTAMP_FMT = "%d.%m.%y, %H:%M:%S"

In [ ]:
# 1. Extract ZIP

with zipfile.ZipFile(whatsapp_backup_path) as zf:
    zf.extractall(whatsapp_backup_extraction_path)

In [ ]:
def _is_blank_message(text: str) -> bool:
    return not text.replace("\u200e", "").strip()

def parse_chat(chat_text: str) -> list[dict]:
    msgs: list[dict] = []
    current: dict | None = None
    for raw in chat_text.splitlines():
        m = BRACKET_LINE_RE.match(raw)
        if m:
            if current is not None:
                if not _is_blank_message(current["content"]):
                    msgs.append(current)
            ts_raw, sender, content = m.groups()
            try:
                ts = _dt.datetime.strptime(ts_raw.strip(), TIMESTAMP_FMT)
            except ValueError:
                raise ValueError(f"Invalid timestamp format: {ts_raw}")
            current = {
                "timestamp": ts,
                "sender": sender.strip(),
                "content": content.replace("\u200e", "").strip(),
                "raw": raw,
            }
        else:
            # Fortsetzung der vorherigen Nachricht (mehrzeilig)
            if current is not None:
                current["content"] += "\n" + raw
    # Letzte Nachricht flushen
    if current is not None and not _is_blank_message(current["content"]):
        msgs.append(current)
    return msgs

chat_file = Path(whatsapp_backup_extraction_path) / "_chat.txt"
if not chat_file.exists():
    raise FileNotFoundError("_chat.txt not found in the ZIP archive.")

chat_text = chat_file.read_text(encoding="utf-8", errors="ignore")
raw_messages = parse_chat(chat_text)

if not raw_messages:
    raise RuntimeError("No messages parsed from _chat.txt")

print(f"Parsed {len(raw_messages)} messages from _chat.txt")

In [ ]:
"""Gruppiert Nachrichten in Conversations nach Senderwechsel oder Anruf."""
def is_call_message(content: str) -> bool:
    lower = content.lower()
    return any(kw in lower for kw in CALL_KEYWORDS)

conversations: list[list[dict]] = []
current = [raw_messages[0]]
for msg in raw_messages[1:]:
    prev_sender = current[-1]["sender"]
    if msg["sender"] != prev_sender or is_call_message(msg["content"]):
        conversations.append(current)
        current = [msg]
    else:
        current.append(msg)
conversations.append(current)

In [ ]:
def convert_messages(
    conv_msgs: list[dict],
    conv_uuid: str,
    extraction_dir: Path,
    attachments_dir: Path,
) -> list[dict]:
    converted: list[dict] = []
    for idx, msg in enumerate(conv_msgs, 1):
        entry: dict = {
            "Id": idx,
            "Text": msg["content"],
            "Timestamp": msg["timestamp"].strftime("%Y-%m-%d %H:%M"),
        }
        text = msg["content"]
        att_matches = list(ATTACHMENT_RE.finditer(text))
        attachments: list[dict] = []
        if att_matches:
            # Platzhalter aus Text entfernen
            for m in att_matches:
                text = text.replace(m.group(0), "")
            entry["Text"] = text.strip()
            for attachment_count, m in enumerate(att_matches, 1):
                filename = m.group(1).strip()
                src = extraction_dir / filename
                if not src.exists():
                    continue  # Datei nicht vorhanden (evtl. Cloud‑only)
                ext = src.suffix.lower()
                tgt_name = f"{conv_uuid}_{idx}_{attachment_count}{ext}"
                dest = attachments_dir / tgt_name
                shutil.copy2(src, dest)
                attachments.append(
                    {"path": tgt_name, "id": attachment_count, "original_filename": filename}
                )
        if attachments:
            entry["Attachments"] = attachments
        converted.append(entry)
    return converted

for idx, conv in enumerate(conversations, 1):
    conv_id = str(uuid.uuid4())
    messages = convert_messages(
        conv, conv_id, Path(whatsapp_backup_extraction_path), Path(attachments_dir)
    )
    start_dt, end_dt = conv[0]["timestamp"], conv[-1]["timestamp"]
    conv_json = {
        "Id": conv_id,
        "channel": "Whatsapp",
        "StartDate": start_dt.strftime("%Y-%m-%d %H:%M"),
        "EndDate": end_dt.strftime("%Y-%m-%d %H:%M"),
        "Sender": conv[0]["sender"],
        "Messages": messages,
    }
    # Save results to JSON file
    output_file = os.path.join(output_dir, f"{conv_id}.json")
    with open(output_file, "w") as f:
        json.dump(conv_json, f, indent=2)
    print(f"Processed whatsapp {idx}/{len(conversations)}: {conv_id}")